# Gemini Sentiment Test

Notebook nay test phan tich cam xuc bang Gemini tren file CSV clean va truc quan ket qua.

# Threads — Cao du lieu & Phan tich Sentiment

Cao 10 bai post / 100 comment theo tu khoa **miule**, sau do phan tich cam xuc bang Gemini.

In [1]:
import asyncio
import sys
from pathlib import Path

import pandas as pd

# Them thu muc goc vao sys.path neu chay tu thu muc thread/
sys.path.insert(0, str(Path('..').resolve()))

from threads.threads_runner import get_threads_data, posts_to_dataframe, comments_to_dataframe
from data_preprocessing import preprocess_threads_comments
from gemini_sentiment import analyze_comments

if sys.platform.startswith('win'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [2]:
# ==== CAU HINH CAO ====
QUERIES            = ['miule']   # Tu khoa tim kiem
POSTS_PER_QUERY    = 3        # So bai post toi da moi query
COMMENTS_PER_POST  = 1000        # So comment toi da moi bai post
CELEBRITY_NAME     = 'Miu Le'    # Ten nghe si (dung trong prompt Gemini)
BATCH_SIZE         =  100          # So comment gui 1 lan cho Gemini
COMMENT_LIMIT      = None        # None = phan tich toan bo

In [3]:
# ==== DOC DU LIEU THREADS TU CSV ====
posts_df = pd.read_csv('threads/miule_posts_raw.csv')
comments_df = pd.read_csv('threads/miule_comments_raw.csv')

print(f'Posts    : {len(posts_df)}')
print(f'Comments : {len(comments_df)}')
comments_df.head()

Posts    : 27
Comments : 2164


,post_url,author_handle,comment_text,create_time
0,https://www.threads.com/@duyvo_vppi/post/DYR9-...,zukuto,zukuto\n2d\nTao nhìn cái Trang còn đẹp hơn Miu...,2026-05-13T15:33:42.000Z
1,https://www.threads.com/@duyvo_vppi/post/DYR9-...,dinhpham5074,dinhpham5074\n1d\nLàm ca mà cũng ngập vậy thì ...,2026-05-14T04:48:42.000Z
2,https://www.threads.com/@duyvo_vppi/post/DYR9-...,hoang1468,hoang1468\n2d\nTrong cơn phê em là nàng công c...,2026-05-13T16:14:30.000Z
3,https://www.threads.com/@duyvo_vppi/post/DYR9-...,hoangphong9012,hoangphong9012\n1d\nMiu Lê nhìn gớm bỏ mẹ.em t...,2026-05-13T22:07:27.000Z
4,https://www.threads.com/@duyvo_vppi/post/DYR9-...,combackh,combackh\n1d\nEm Trang nằm vùng để bắt tội phạ...,2026-05-13T23:53:22.000Z


In [4]:
# ==== TIEN XU LY ====
comments_clean_df = preprocess_threads_comments(comments_df)
print(f'Comments sau xu ly: {len(comments_clean_df)}')
comments_clean_df.head()

Comments sau xu ly: 2163


,post_url,author_handle,comment_text,create_time,comment_text_raw
0,https://www.threads.com/@duyvo_vppi/post/DYR9-...,zukuto,zukuto\nTao nhìn cái Trang còn đẹp hơn Miu Lê,2026-05-13 15:33:42+00:00,zukuto\n2d\nTao nhìn cái Trang còn đẹp hơn Miu...
1,https://www.threads.com/@duyvo_vppi/post/DYR9-...,dinhpham5074,dinhpham5074\nLàm ca mà cũng ngập vậy thì còn ...,2026-05-14 04:48:42+00:00,dinhpham5074\n1d\nLàm ca mà cũng ngập vậy thì ...
2,https://www.threads.com/@duyvo_vppi/post/DYR9-...,hoang1468,hoang1468\nTrong cơn phê em là nàng công chúa....,2026-05-13 16:14:30+00:00,hoang1468\n2d\nTrong cơn phê em là nàng công c...
3,https://www.threads.com/@duyvo_vppi/post/DYR9-...,hoangphong9012,hoangphong9012\nMiu Lê nhìn gớm bỏ mẹ.em trang...,2026-05-13 22:07:27+00:00,hoangphong9012\n1d\nMiu Lê nhìn gớm bỏ mẹ.em t...
4,https://www.threads.com/@duyvo_vppi/post/DYR9-...,combackh,combackh\nEm Trang nằm vùng để bắt tội phạm đấy,2026-05-13 23:53:22+00:00,combackh\n1d\nEm Trang nằm vùng để bắt tội phạ...


In [5]:
# ==== PHAN TICH SENTIMENT ====
sentiment_df = analyze_comments(
    comments_clean_df,
    text_column='comment_text',
    celebrity_name=CELEBRITY_NAME,
    batch_size=BATCH_SIZE,
    limit=COMMENT_LIMIT,
)

print(f'Ket qua: {len(sentiment_df)} dong')
sentiment_df[['comment_text','sentiment','emotion','topic','controversy','confidence','rationale']].head(10)

[*] Cần xử lý tiếp: 2163 records.
  -> Đã lưu tạm 100 kết quả...
  -> Đã lưu tạm 200 kết quả...
  -> Đã lưu tạm 300 kết quả...
  -> Đã lưu tạm 400 kết quả...
  -> Đã lưu tạm 500 kết quả...
  -> Đã lưu tạm 600 kết quả...
  -> Đã lưu tạm 700 kết quả...
  -> Đã lưu tạm 800 kết quả...
  -> Đã lưu tạm 900 kết quả...
  -> Đã lưu tạm 1000 kết quả...
  -> Đã lưu tạm 1100 kết quả...
  -> Đã lưu tạm 1200 kết quả...
  -> Đã lưu tạm 1300 kết quả...
  -> Đã lưu tạm 1400 kết quả...

[!] Dừng giữa chừng do lỗi: Unterminated string starting at: line 1 column 8326 (char 8325)
[*] Không sao, đã lưu an toàn 1400 kết quả vào sentiment_checkpoint.csv.
[*] Bạn chỉ cần chạy lại Cell này, nó sẽ tự động chạy tiếp phần còn thiếu.
Ket qua: 2163 dong


,comment_text,sentiment,emotion,topic,controversy,confidence,rationale
0,zukuto\nTao nhìn cái Trang còn đẹp hơn Miu Lê,negative,tuc_gian,appearance,True,0.90,So sanh ngoai hinh Miu Le voi nguoi khac mot c...
1,dinhpham5074\nLàm ca mà cũng ngập vậy thì còn ...,negative,that_vong,drama,True,0.90,Bay to su that vong va yeu cau xu ly nghiem ca...
2,hoang1468\nTrong cơn phê em là nàng công chúa....,negative,tuc_gian,drama,True,0.85,"Mo ta cam giac khi choi ma tuy, lien quan den ..."
3,hoangphong9012\nMiu Lê nhìn gớm bỏ mẹ.em trang...,negative,tuc_gian,appearance,True,0.90,Che ngoai hinh Miu Le va bay to su that vong v...
4,combackh\nEm Trang nằm vùng để bắt tội phạm đấy,neutral,hai_huoc,rumor,True,0.80,"Binh luan mang tinh troll, cho rang ""Em Trang""..."
5,dzng_trzn\nGhép ảnh bậy bạ bi bế lúc nào ko hay,neutral,trung_lap,social_post,True,0.80,Canh bao ve viec ghep anh bay ba co the bi xu ...
6,thanhrau94\nẢnh ghép nhưng mà tội là thật. Đăn...,negative,tuc_gian,drama,True,0.85,"Phan bac viec anh ghep, khang dinh toi that va..."
7,"nguyenquyken\nChắc không phải CA đâu. Mà là ""n...",neutral,trung_lap,rumor,True,0.75,Nghi ngo nguoi trong anh khong phai cong an th...
8,hai.trieu.vuong.68\nCông an mà xử ma túy thì c...,negative,that_vong,drama,True,0.85,"Bay to su bat luc, that vong khi cong an lien ..."
9,"taquang87\nNó thích thì nó dùng, nó dùng thì n...",neutral,trung_lap,drama,True,0.80,Bay to quan diem ai dung ma tuy thi phai chiu ...


In [9]:
# ==== LUU CSV ====
import os

out_dir = Path('threads')
out_dir.mkdir(exist_ok=True)

tag = '_'.join(q.lower().replace(' ', '_') for q in QUERIES)
posts_path     = out_dir / f'{tag}_posts.csv'
comments_path  = out_dir / f'{tag}_comments.csv'
sentiment_path = out_dir / f'{tag}_comments_sentiment.csv'

posts_df.to_csv(posts_path, index=False, encoding='utf-8-sig')
comments_clean_df.to_csv(comments_path, index=False, encoding='utf-8-sig')
sentiment_df.to_csv(sentiment_path, index=False, encoding='utf-8-sig')

print(f'Posts    -> {posts_path}')
print(f'Comments -> {comments_path}')
print(f'Sentiment-> {sentiment_path}')

Posts    -> threads\miule_posts.csv
Comments -> threads\miule_comments.csv
Sentiment-> threads\miule_comments_sentiment.csv
